# Módulo 5 — Textura (GLCM) y criterio D

| | |
|---|---|
| **Responde** | Extracción de características (texturas) · metodología 6 |
| **Código** | `textura_criterio_d.py` (en esta misma carpeta) |
| **Recibe** | la imagen mejorada y la máscara |
| **Devuelve** | atributos GLCM; D estimado (0-5) |
| **Responsable** | Lucas D'Elia |

## Qué hace este módulo

En la regla ABCD, D cuenta cuántas **estructuras diferenciales** ve el dermatólogo dentro de la lesión. Son los dibujos o patrones que se ven con el dermatoscopio, y se llaman así porque ayudan a diferenciar un lunar común de uno sospechoso. PH2 anota cinco.

- **Red de pigmento.** Líneas marrones finas que forman una malla, parecida a un panal. Si es pareja se llama típica y es común en lunares benignos; si es irregular se llama atípica y se asocia al melanoma.
- **Puntos y glóbulos.** Manchitas redondas y oscuras. Los puntos son muy chicos y los glóbulos, un poco más grandes.
- **Estrías.** Rayitas que salen del borde hacia afuera, como rayos.
- **Áreas de regresión.** Zonas donde el pigmento desapareció, que se ven blancas como una cicatriz o con puntitos grises azulados.
- **Velo azul-blanquecino.** Una mancha azulada y difusa con una capa blanca encima, como vista a través de un vidrio esmerilado.

**Por qué D es adaptado.** La regla original cuenta otras cinco estructuras (red de pigmento, áreas sin estructura, estrías, puntos y glóbulos) y va de 1 a 5. PH2 trae los puntos y glóbulos juntos, no anota las áreas sin estructura y suma la regresión y el velo, así que con sus anotaciones no se puede medir el D original. Por eso definimos un **D adaptado**, que cuenta cuántas de las cinco estructuras de PH2 marcó el dermatólogo y va de 0 a 5. La red de pigmento sólo cuenta cuando es atípica, porque en PH2 todas las lesiones tienen red y su mera presencia no distingue nada.

Como este D se mide distinto, el umbral clínico del TDS (4,75), que se fijó con el D original, puede no funcionar igual. Por eso M7 lo compara con un umbral ajustado a nuestros datos.

La **textura** de la lesión es la base de varias de esas medidas. La matriz de co-ocurrencia de niveles de gris (GLCM) cuenta cuántas veces aparece cada nivel de gris al lado de otro, a una distancia y en una dirección dadas. En una lesión lisa los pares se concentran en niveles parecidos, y en una con mucho patrón se reparten. De la matriz salen siete atributos (energía, entropía, contraste, IDM, correlación, *cluster shade* y *cluster prominence*), que se calculan sobre el brillo V de la imagen mejorada y usando sólo pares de píxeles dentro de la lesión.

Para estimar D hacemos lo mismo que el dermatólogo. Buscamos cada estructura con una medida pensada para ella y contamos cuántas encontramos (sección 2).

El módulo recibe la imagen mejorada por M1 y la máscara de M3, y devuelve los siete atributos de cada imagen, una medida por estructura y, una vez ajustados los umbrales, el D estimado. Todo este cuaderno usa sólo las 140 imágenes de entrenamiento; la evaluación en las 60 de prueba está en el cuaderno de M7.

In [1]:
# Permite usar el código de las otras carpetas del repo
import sys
sys.path.insert(0, "../..")

# Recarga los cambios que hagas en los archivos .py sin reiniciar el cuaderno
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from modulos.m0_carga_de_datos import carga_de_datos
from modulos.m1_mejoramiento import mejoramiento
from modulos.m3_segmentacion import segmentacion
from modulos.m5_textura_criterio_d import textura_criterio_d
from modulos.m7_clasificacion_resultados import clasificacion_resultados as m7
from modulos.herramientas import graficos

graficos.aplicar_estilo()

## 1. Medidas en las imágenes de entrenamiento

La GLCM tiene dos parámetros. Los **niveles de gris** son en cuántos valores se divide el brillo, y usamos 32, que conservan detalle sin que la matriz de 32 × 32 quede casi vacía. La **distancia** es cuántos píxeles separan a los dos elementos de cada par, y usamos 1, que compara píxeles vecinos, la escala de las líneas finas de la red de pigmento.

La celda siguiente mejora y segmenta cada imagen de entrenamiento con M1 y M3, que es lo mismo que hace el algoritmo, y calcula los siete atributos de la GLCM y la medida de cada estructura. También guarda qué estructuras marcó el dermatólogo, para elegir los umbrales y comprobar los detectores.

In [2]:
# Para cada imagen de entrenamiento: mejora, segmenta con M3 y calcula la textura y las medidas de cada estructura
filas = []
for nombre in carga_de_datos.nombres("entrenamiento"):
    caso = carga_de_datos.cargar_caso(nombre)
    img, valida, _ = mejoramiento.mejorar(caso.imagen)
    mascara = segmentacion.segmentar(img, valida)
    v = mejoramiento.a_hsv(img)[..., 2]
    fila = {"nombre": nombre, "D_derm": caso.d_dermatologo, "diagnostico": caso.diagnostico, "sospechosa": caso.sospechosa,
            "dice": segmentacion.dice(mascara, caso.mascara_ref)}
    fila.update({f"glcm_{k}": x for k, x in textura_criterio_d.atributos_glcm(v, mascara).items()})
    fila.update({f"estr_{k}": x for k, x in textura_criterio_d.medidas_estructuras(img, mascara).items()})
    fila.update({f"derm_{k}": x for k, x in caso.estructuras.items()})
    filas.append(fila)
tabla = pd.DataFrame(filas)
print("Dice de M3 contra la referencia: mediana", round(tabla["dice"].median(), 3), "| mínimo", round(tabla["dice"].min(), 3))
tabla[[f"glcm_{k}" for k in textura_criterio_d.ATRIBUTOS]].describe().round(3)

Dice de M3 contra la referencia: mediana 0.931 | mínimo 0.501


,glcm_energia,glcm_entropia,glcm_contraste,glcm_idm,glcm_correlacion,glcm_cluster_shade,glcm_cluster_prominence
count,140.000,140.000,140.000,140.000,140.000,140.000,140.000
mean,0.041,5.567,1.223,0.680,0.922,31.205,8142.637
std,0.052,0.711,0.334,0.047,0.069,268.730,8714.898
min,0.014,2.604,0.450,0.584,0.602,-813.161,69.761
25%,0.021,5.290,0.969,0.650,0.897,-84.283,1679.428
50%,0.029,5.644,1.177,0.679,0.948,-11.184,4746.685
75%,0.039,6.083,1.429,0.704,0.966,115.902,12354.411
max,0.470,6.649,2.326,0.891,0.989,965.875,46968.258


## 2. Criterio D por estructuras

Buscamos cada estructura por separado, con una medida que sale de cómo se ve.

- **Red atípica.** Una red regular tiene un contraste local parecido en toda la lesión, y una atípica lo hace variar. Medimos cuánto varía el mapa de contraste de M6 (desvío sobre media).
- **Puntos y glóbulos.** Son manchas oscuras y redondeadas. Con un *black-hat* (la misma operación de los pelos en M1) quedan las manchas oscuras chicas, una apertura borra las líneas finas de la red, y contamos las manchas por cada 10 000 píxeles de lesión.
- **Estrías.** Son líneas oscuras y finas que salen del borde hacia afuera. Con la matriz Hessiana marcamos las líneas oscuras y su dirección, y medimos qué parte de un anillo de 15 px a cada lado del borde tiene líneas marcadas que apuntan hacia afuera.
- **Regresión.** Son zonas blancas, más claras y menos saturadas que la piel. Como en el canal azul se parecen a la piel, M3 suele dejarlas afuera de la máscara, así que las buscamos en toda la envolvente convexa de la lesión.
- **Velo.** Es la única estructura azulada. Medimos cuánto domina el azul sobre el rojo en la parte más azulada de la lesión (percentil 90 de B − R).

Cada medida es un solo número por imagen, y una estructura se considera presente si supera un umbral. El umbral se elige con el **índice de Youden**, J = sensibilidad + especificidad − 1. La sensibilidad es la fracción de imágenes con la estructura que el detector encuentra, y la especificidad, la fracción de imágenes sin la estructura que deja sin marcar. Se prueban todos los umbrales posibles y se elige el de mayor J, que es el que mejor equilibra no perder estructuras y no inventarlas. El D estimado es la cantidad de estructuras detectadas, de 0 a 5.

La tabla siguiente muestra, para cada estructura, en cuántas imágenes de entrenamiento la marcó el dermatólogo, qué tan bien la separa su medida y el umbral elegido. Lo que separa se mide con el **AUC**, que es la probabilidad de que una imagen con la estructura tenga una medida más alta que una imagen sin ella. Vale 0,5 si la medida no distingue nada, como tirar una moneda, y 1 si las separa perfectamente. El umbral está en las unidades de cada medida (por ejemplo, manchas cada 10 000 px en los glóbulos), así que los umbrales no se comparan entre sí.

In [3]:
# Qué tan bien separa cada medida las imágenes con y sin esa estructura, y su umbral
umbrales_e = textura_criterio_d.umbrales_estructuras(tabla)
por_estructura = pd.DataFrame([{"estructura": e, "imágenes con la estructura": int(tabla[f"derm_{e}"].sum()),
                                "AUC": m7.auc(tabla[f"estr_{e}"].fillna(tabla[f"estr_{e}"].min()), tabla[f"derm_{e}"].to_numpy(bool)),
                                "umbral": umbrales_e[e]} for e in textura_criterio_d.ESTRUCTURAS])
graficos.guardar_tabla(por_estructura, "m5_estructuras", decimales=2)
por_estructura.round(3)

,estructura,imágenes con la estructura,AUC,umbral
0,red_pigmento,82,0.664,0.390
1,puntos_globulos,78,0.603,15.346
2,estrias,18,0.785,0.028
3,regresion,19,0.787,0.001
4,velo,24,0.945,-0.161


Con esos umbrales, el D estimado de cada imagen es la cantidad de estructuras detectadas. Para medir qué tan bien funciona sin usar las mismas imágenes con las que se eligieron los umbrales usamos **validación cruzada**. Partimos las 140 imágenes en 4 partes con proporciones parecidas de cada D, elegimos los umbrales con 3 partes y contamos las estructuras en la que quedó afuera, rotando hasta pasar por las 4. Repetimos todo con 20 sorteos y promediamos.

La tabla resume cuatro números. El **acierto exacto** es la fracción de imágenes en que el D estimado coincide con el del dermatólogo, y el **error medio** es cuántos puntos de D se equivoca en promedio. El **κ de Cohen** mide la coincidencia descontando la que se daría por azar, porque como la mayoría de las imágenes tiene D igual a 1 o 2, se puede acertar bastante sin medir nada; vale 1 si el acuerdo es perfecto y 0 si es el esperable por azar. El **AUC** es el mismo de la tabla anterior, pero ahora mide si el D estimado separa las lesiones sospechosas (nevos atípicos y melanomas) de los nevos comunes, que es lo que le importa al TDS.

In [4]:
def partes_estratificadas(grupos, partes, semilla):
    """Reparte las imágenes en partes con proporciones de grupos parecidas.

    Entra: grupos   array con el grupo de cada imagen
           partes   cantidad de partes
           semilla  entero, para que dé siempre lo mismo
    Sale : array int con el número de parte de cada imagen
    """
    rng = np.random.default_rng(semilla)
    parte = np.zeros(len(grupos), int)
    for grupo in np.unique(grupos):
        indices = rng.permutation(np.flatnonzero(grupos == grupo))
        parte[indices] = (np.arange(len(indices)) + rng.integers(partes)) % partes
    return parte

# Validación cruzada. En cada repetición se reparten las imágenes en 4 partes; se ajustan
# los umbrales con 3 y se cuentan las estructuras en la que quedó afuera, hasta pasar por las 4
y, sospechosa = tabla["D_derm"].to_numpy(), tabla["sospechosa"].to_numpy()
filas = []
for semilla in range(20):
    estimado = np.zeros(len(y), int)
    parte = partes_estratificadas(np.minimum(y, 3), 4, semilla)
    for k in range(4):
        u = textura_criterio_d.umbrales_estructuras(tabla[parte != k])
        estimado[parte == k] = [sum(textura_criterio_d.detectar_estructuras({e: fila[f"estr_{e}"] for e in u}, u).values())
                                for _, fila in tabla[parte == k].iterrows()]
    filas.append({"acierto exacto": (estimado == y).mean(), "error medio": np.abs(estimado - y).mean(),
                  "kappa": m7.kappa_cohen(estimado, y), "AUC sospechosa": m7.auc(estimado, sospechosa)})
referencia = (y == np.median(y)).mean()
print("Decir siempre la mediana de D acierta en", round(float(referencia), 3), "| D del dermatólogo: AUC sospechosa", round(m7.auc(y, sospechosa), 3))
pd.DataFrame(filas).mean().round(3)

Decir siempre la mediana de D acierta en 0.371 | D del dermatólogo: AUC sospechosa 0.924


acierto exacto    0.276
error medio       1.139
kappa             0.072
AUC sospechosa    0.728
dtype: float64

El velo es la estructura que mejor se detecta (AUC 0,95), seguido por la regresión y las estrías (0,79). La red atípica (0,66) y los puntos y glóbulos (0,60) se detectan peor, porque son patrones finos que una foto de este tamaño apenas resuelve. El umbral de regresión quedó casi en cero, así que cuenta como presente cualquier zona blanca, por chica que sea.

Con validación cruzada, el D estimado acierta el valor exacto en el 28 % de las imágenes, menos que decir siempre la mediana (37 %), y su acuerdo con el dermatólogo es bajo (κ = 0,07). Pero separa las lesiones sospechosas de las comunes con un AUC de 0,73 (el D del dermatólogo da 0,92), y eso es lo que importa para el TDS. Acertar el valor exacto es difícil porque basta errar en una estructura para errar el número.

La evaluación en las 60 imágenes de prueba está en el cuaderno de M7.